In [19]:
import numpy as np
import torch
import torchvision.transforms as T
#from decord import VideoReader, cpu
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
from pathlib import Path
import re
import json
import tqdm
import os

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values).to(torch.bfloat16).cuda()
    return pixel_values

In [2]:
open_ai_scene_types = {
    ("apartment",): "Apartment",
    ("bathroom",): "Bathroom",
    ("bedroom", "hotel"): "Bedroom / Hotel",
    ("hotel", "bedroom"): "Bedroom / Hotel",
    ("bookstore", "library"): "Bookstore / Library",
    ("library", "bookstore"): "Bookstore / Library",
    ("classroom",): "Classroom",
    ("conference", "room"): "Conference Room",
    ("copy", "mail", "room"): "Copy / Mail Room",
    ("mail", "copy", "room"): "Copy / Mail Room",
    ("kitchen",): "Kitchen",
    ("laundry",): "Laundry Room",
    ("living", "room", "lounge"): "Living room / Lounge",
    ("lounge", "living", "room"): "Living room / Lounge",
    ("lobby",): "Lobby",
    ("office",): "Office",
    ("storage", "basement", "garage"): "Storage / Basement / Garage",
    ("storage", "garage", "basement"): "Storage / Basement / Garage",
    ("basement", "storage", "garage"): "Storage / Basement / Garage",
    ("basement", "garage", "storage"): "Storage / Basement / Garage",
    ("garage", "storage", "basement"): "Storage / Basement / Garage",
    ("garage", "basement", "storage"): "Storage / Basement / Garage",
}

In [4]:
DEFAULT_DATA_DIR: Path = Path("./") / "prompts_sceneType"

PROMPT_NAME_TO_PATH = {
    "blind": DEFAULT_DATA_DIR / Path("blind.txt"),
    "blind_not_step_by_step": DEFAULT_DATA_DIR / Path("blind_not_step_by_step.txt"),
    "vision": DEFAULT_DATA_DIR / Path("vision.txt"),
    "vision_not_step_by_step": DEFAULT_DATA_DIR / Path("vision_not_step_by_step.txt"),
    "vision_and_text_prefix": DEFAULT_DATA_DIR / Path("vision_and_text_prefix.txt"),
    "vision_and_text_prefix_not_step_by_step": DEFAULT_DATA_DIR / Path("vision_and_text_prefix_not_step_by_step.txt"),
    "vision_and_text_suffix": DEFAULT_DATA_DIR / Path("vision_and_text_suffix.txt"),
}

def load_prompt(name: str):
    if name not in PROMPT_NAME_TO_PATH:
        raise ValueError("invalid prompt: {}".format(name))
    path = PROMPT_NAME_TO_PATH[name]
    with path.open("r") as f:
        return f.read().strip()

In [5]:
import json
# load dataset
dataset = {}
for item in json.load(open("data/open-eqa-v0_sceneType.json", "r", encoding="utf-8")):
    if "sceneType" in item:
        episode_history = item["episode_history"]
        if not episode_history in dataset:
            dataset[episode_history] = {"questions":[], "answers":[], "sceneType": item["sceneType"]}
        dataset[episode_history]["questions"].append(item["question"])
        dataset[episode_history]["answers"].append(item["answer"])
print("found {:,} episode histories".format(len(dataset)))

found 79 episode histories


In [6]:
model_name = "OpenGVLab/InternVL2_5-38B"
seed = 1234
max_tokens = 4000
temperature = 0.2
image_size = 512
num_q_and_a = 14
prompts = ["blind", "blind_not_step_by_step"]
if "vision" in model_name or "modal" in model_name or "VL" in model_name:
    prompts.extend(["vision", "vision_not_step_by_step", "vision_and_text", "vision_and_text_not_step_by_step"])

In [18]:
model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=max_tokens, temperature=temperature, do_sample=True)

In [ ]:
for prompt in prompts:
    results = []
    freq = {}
    for idx, (episode_history, item) in enumerate(tqdm.tqdm(list(dataset.items()))):
        # get Q&A
        questions_and_answers = []
        for question, answer in zip(item["questions"], item["answers"]):
            questions_and_answers.append(f"Q: {question}\nA: {answer}")
            if len(questions_and_answers) == num_q_and_a:
                break
        questions_and_answers = "\n\n".join(questions_and_answers)
        
        # extract scene paths and prepare the prompt
        text = ""
        pixel_values = None
        if "vision" in prompt:
            pixel_values = load_image(os.path.join("data/scene_images", f"{episode_history}.png"))
            
            suffix = None
            if "text" in prompt:
                if "not_step_by_step" in prompt:
                    text = load_prompt("vision_and_text_prefix_not_step_by_step")
                else:
                    text = load_prompt("vision_and_text_prefix")
                suffix = load_prompt("vision_and_text_suffix")
                suffix = suffix.format(questions_and_answers=questions_and_answers)
                text = "\n\n".join([text, suffix])
            else:
                if "not_step_by_step" in prompt:
                    text = load_prompt("vision_not_step_by_step")
                else:
                    text = load_prompt("vision")
            text = "\n".join(["<image>\n", text])
        else:
            text = load_prompt(prompt).format(questions_and_answers=questions_and_answers)

        torch.manual_seed(seed)
        output = model.chat(tokenizer, pixel_values, text, generation_config)
        lower_output = output.lower()
        if "the category is" in lower_output:
            lower_output = re.sub(".*the category is", "", lower_output, flags=re.MULTILINE | re.DOTALL)
            words = [re.sub("\W+", "", word) for word in lower_output.split()]
            words = [word for word in words if word and word != "or"]
        else:
            words = [re.sub("\W+", "", word) for word in lower_output.split()]
            words = [word for word in words if word and word != "or"]
        estimated_sceneType = None
        for i in range(len(words)):
            for j in range(1, 4):
                if i + j == len(words) + 1:
                    break
                scene_type_key = tuple(words[i:i+j])
                if scene_type_key in open_ai_scene_types:
                    estimated_sceneType = open_ai_scene_types[scene_type_key]
                    break
            if estimated_sceneType:
                break
        
        # count sceneTypes
        if not item["sceneType"] in freq:
            freq[item["sceneType"]] = [0, 0]
        freq[item["sceneType"]][0] += 1
        if item["sceneType"] == estimated_sceneType:
            freq[item["sceneType"]][1] += 1

        # store results
        results.append({
            "episode_history": episode_history,
            "output": output,
            "estimated_sceneType": estimated_sceneType,
            "sceneType": item["sceneType"]
        })
        json.dump(results, open("data/results/{}-{}-{}.json".format(re.sub(r".*/", "", model_name), prompt, seed), "w"), indent=2)

    # save at end (redundant)
    json.dump(results, open("data/results/{}-{}-{}.json".format(re.sub(r".*/", "", model_name), prompt, seed), "w"), indent=2)
    print("saving {:,} answers".format(len(results)))
    
    for k in ["Apartment", "Bathroom", "Bedroom / Hotel", "Bookstore / Library", "Classroom", "Conference Room", "Copy / Mail Room", "Kitchen", "Laundry Room", "Living room / Lounge", "Lobby", "Office", "Storage / Basement / Garage"]:
        print(k, freq[k])
        

In [15]:
output

['The category is (Library)']